# Round 6 — Local support-density evidence

One bounded feature study on existing Qwen caches. Twelve new CPU readouts; no new encoding, model download, neural training, score blend, or submission. The 881-comment development cohort is repeatedly inspected, not an independent holdout.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/local_support_features.json").is_file())
from scripts.run_local_support_features import figures, write_dashboard
r = json.loads((ROOT / "reports/local_support_features/results.json").read_text())
CHARTS = figures(r)
print("Run:", r["run_id"], "| New fits at completion:", r["new_fits"])
print("Decision:", r["decision"])
print("Primary:", r["identity"]["config"]["primary"])

Run: 32a90cf7c6607257c2bf | New fits at completion: 12
Decision: DO_NOT_PROMOTE_PRIMARY
Primary: frozen_all


## 1. Why a different semantic feature investigation?

The prior audit found lexical readouts slightly stronger on advertising, but Qwen stronger on legal advice. Centroid and nearest-example primitives were already investigated. New local density and evidence concentration must beat a historical-geometry control, not merely a weaker lexical model.

In [2]:
prior = json.loads((ROOT / "reports/feature_value_audit/audit.json").read_text())
display(pd.DataFrame(prior["metrics"]))
display(pd.DataFrame(r["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")

,fold,policy,model,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_adapted_reference,0.679254,0.250238,0.760003
1,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527
2,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719
3,0,"No Advertising: Spam, referral links, unsolici...",add_act_roles,0.675784,0.231472,0.663380
4,0,"No Advertising: Spam, referral links, unsolici...",condition_lexical,0.686604,0.231335,0.669720
5,0,"No Advertising: Spam, referral links, unsolici...",rule_both,0.688694,0.231786,0.670448
6,0,"No Advertising: Spam, referral links, unsolici...",copy_both,0.695410,0.232222,0.678693
7,0,"No Advertising: Spam, referral links, unsolici...",scope_both,0.695485,0.232151,0.678278
8,1,No legal advice: Do not offer or request legal...,qwen_adapted_reference,0.760533,0.221528,0.709879
9,1,No legal advice: Do not offer or request legal...,lexical_control,0.641993,0.232466,0.656063


,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",frozen_local,0.678545,0.257715,0.828465
7,1,No legal advice: Do not offer or request legal...,frozen_local,0.750416,0.232593,0.804161
8,0,"No Advertising: Spam, referral links, unsolici...",frozen_all,0.683172,0.253272,0.822787
9,1,No legal advice: Do not offer or request legal...,frozen_all,0.751854,0.233589,0.814623


## 2. Training-only support features

A reference pool contains explicitly labeled examples under one rule. Three normalized-text group folds exclude each training example from its own reference pool. Query features use only the full eligible training pool. There are no query-query distances, query targets, or inferred cross-rule negatives.

In [3]:
catalog = json.loads((ROOT / "reports/local_support_features/feature_catalog.json").read_text())
print("Historical basic feature count:", len(catalog["historical_basic"]))
print("New density feature count:", len(catalog["new_local_density"]))
CHARTS[4].show(renderer="plotly_mimetype")

Historical basic feature count: 9
New density feature count: 12


## 3. Local affinity evidence

Distances are scaled by the local reference and query radii. Class mean affinities, class contrasts, entropy and effective-support fractions describe how much conflicting evidence surrounds the comment. These quantities are not asserted to be calibrated probabilities.

In [4]:
CHARTS[5].show(renderer="plotly_mimetype")
print("Features are constructed without evaluation labels.")

Features are constructed without evaluation labels.


## 4. Measured contribution and uncertainty

The matched answer-only readout controls the change in output layer. Basic geometry controls centroid, nearest and top-five similarity. Removing each family measures conditional value. Intervals cover 11 fixed contrasts but not historical adaptive selection.

In [5]:
display(pd.DataFrame(r["comparisons"]))
CHARTS[1].show(renderer="plotly_mimetype")
CHARTS[2].show(renderer="plotly_mimetype")

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,answer_only,qwen_raw,answer_only vs raw Qwen,0.000000,-0.016891,0.016891
1,frozen_basic,qwen_raw,frozen_basic vs raw Qwen,0.003174,-0.013716,0.020065
2,frozen_local,qwen_raw,frozen_local vs raw Qwen,-0.005413,-0.022304,0.011477
3,frozen_all,qwen_raw,frozen_all vs raw Qwen,-0.002380,-0.019271,0.014510
4,shuffled_all,qwen_raw,shuffled_all vs raw Qwen,-0.005977,-0.022868,0.010913
5,adapted_all,qwen_raw,adapted_all vs raw Qwen,0.000102,-0.016788,0.016993
6,frozen_all,answer_only,All geometry beyond answer-only readout,-0.002380,-0.019271,0.014510
7,frozen_all,frozen_basic,Local density beyond historical geometry,-0.005555,-0.022445,0.011336
8,frozen_all,frozen_local,Basic geometry beyond local density,0.003033,-0.013858,0.019923
9,frozen_all,shuffled_all,Explicit support labels versus shuffled labels,0.003597,-0.013294,0.020487


## 5. Label and representation controls

Reference labels are shuffled only within each eligible reference pool. The primary uses frozen vectors; adapted vectors are a sensitivity arm. Adapted training scores/vectors are not out-of-fold encoder predictions. Cross-fitting support statistics does not remove this limitation.

In [6]:
CHARTS[3].show(renderer="plotly_mimetype")
CHARTS[6].show(renderer="plotly_mimetype")

## 6. Probability quality and metric scope

AUC is measured on margins, not saturated probabilities. Brier and log loss use probabilities. This is not a direct comparison with the 0.91425 recorded Kaggle private score, nor a claim to exceed the leaderboard.

In [7]:
display(pd.DataFrame(r["pooled_metrics"]))
CHARTS[7].show(renderer="plotly_mimetype")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,frozen_local,0.714480,0.732433,0.731289
4,frozen_all,0.717513,0.733657,0.733572
5,shuffled_all,0.713916,0.727191,0.726985
6,adapted_all,0.719996,0.742625,0.741000


## 7. Fixed decision and saved evidence

Primary frozen_all must beat raw Qwen, answer_only, frozen_basic and shuffled_all by >=0.003 macro AUC with positive simultaneous lower bounds and no policy regression. Ranked-pooled AUC cannot fall versus Qwen. Passing is eligibility for more validation only. No secondary winner is silently promoted. [Local scaling](https://proceedings.neurips.cc/paper/2004/hash/40173ea48d9567f1f393b20c855bb40b-Abstract.html) and [Rule By Example](https://aclanthology.org/2023.acl-long.22/) motivate research, not a claim of reproducing those models.

In [8]:
for item in r["primary_requirements"]:
    display(pd.DataFrame([item["contrast"]]))
    print(item["reference"], item["per_policy_delta"], item["passed"])
print("Decision:", r["decision"])
for item in r["limitations"]:
    print(item)
print("Dashboard:", write_dashboard(ROOT, r))

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,frozen_all,qwen_raw,frozen_all vs raw Qwen,-0.00238,-0.019271,0.01451


qwen_raw [0.003917910447761241, -0.008678890824054153] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,frozen_all,answer_only,All geometry beyond answer-only readout,-0.00238,-0.019271,0.01451


answer_only [0.003917910447761241, -0.008678890824054153] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,frozen_all,frozen_basic,Local density beyond historical geometry,-0.005555,-0.022445,0.011336


frozen_basic [-0.009925373134328397, -0.0011839298643862017] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,frozen_all,shuffled_all,Explicit support labels versus shuffled labels,0.003597,-0.013294,0.020487


shuffled_all [-0.0028358208955223674, 0.010029157942114675] False
Decision: DO_NOT_PROMOTE_PRIMARY
Repeatedly inspected two-policy development cohort, not fresh validation.
Primary geometry uses frozen vectors; reference statistics are cross-fitted.
Adapted sensitivity vectors and answer scores are in-sample on training supports.
Cross-fitting support statistics is not out-of-fold encoder training.
Inner reference pools are smaller than the full outer prediction pool.
Primary is fixed; secondary winners do not replace it post hoc.
No contrastive encoder, routing rule, or ensemble blend is trained.
Conditional intervals cover this round only, not historical adaptive choices.


Dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/local_support_features/dashboard.html
